# Custom Squat Video Augmentation Notebook

## 1. Project Context
PhysioVision AI uses conservative augmentation for robustness experiments. Augmented data is synthetic, is not independent clinical evidence, and does not replace additional consented real videos or physiotherapist label review.

In [3]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
from augment_custom_squat_videos import AUGMENTATION_PLAN, generate_augmentations, augment_frame

RAW_DIR = REPO_ROOT / 'data/raw/custom_videos'
OUTPUT_DIR = REPO_ROOT / 'data/augmented/custom_videos'
METADATA_PATH = REPO_ROOT / 'data/processed/labels/augmented_custom_squat_videos_labels.csv'
MAX_AUGMENTATIONS_PER_SOURCE = 2

## 2. Dataset Overview
Load current metadata, show counts, and flag the severe imbalance before generating anything.

In [4]:
labels_path = REPO_ROOT / 'data/processed/labels/custom_squat_videos_labels.csv'
labels = pd.read_csv(labels_path)
supported = labels[labels['is_supported_video'].astype(str).str.lower() == 'true']
display(supported['label'].value_counts().rename_axis('label').to_frame('videos'))
print(f'Total supported videos: {len(supported)}')
print('WARNING: class counts are small and imbalanced; augmentation does not create independent samples.')

,videos
label,
squat_correct,13
squat_knee_valgus,4
squat_trunk_lean,4
squat_shallow_depth,2
unlabeled,1


Total supported videos: 24


## 3. Augmentation Strategy
Safe operations are mild brightness/contrast changes, horizontal flip with left/right warnings, rotations within 5 degrees, full-frame-preserving zoom-out, mild noise, compression variation, and documented speed variation only within the existing fast/uncontrolled label. Unsafe operations include heavy crops, distortion, stretching, large rotation, joint warping, or changing a correct label into pathology-like form.

## 4. Utility Functions
Reusable loading, frame transformation, output safety, video writing, metadata probing, and the augmentation registry are implemented in `scripts/augment_custom_squat_videos.py` so notebook and CLI behavior remain identical and testable.

In [5]:
AUGMENTATION_PLAN

{'squat_correct': ['brightness', 'contrast', 'horizontal_flip'],
 'squat_shallow_depth': ['brightness',
  'contrast',
  'horizontal_flip',
  'slight_rotation'],
 'squat_knee_valgus': ['brightness',
  'contrast',
  'horizontal_flip',
  'slight_rotation'],
 'squat_trunk_lean': ['brightness',
  'contrast',
  'horizontal_flip',
  'slight_rotation'],
 'squat_fast_uncontrolled': ['brightness', 'contrast']}

## 5. Augmentations
The helper implements horizontal flip, brightness, contrast, slight rotation, slight zoom with reflected borders, Gaussian noise, speed variation, and MP4 compression variation without overwriting source files.

## 6. Generate Augmented Videos
Review `AUGMENTATION_PLAN` and the maximum first. Running the next cell writes local ignored videos and a metadata registry.

In [6]:
# Uncomment only after reviewing the plan and available disk space.
# rows = generate_augmentations(RAW_DIR, OUTPUT_DIR, METADATA_PATH, MAX_AUGMENTATIONS_PER_SOURCE)
# print(f'Generated {len(rows)} augmented videos')

## 7. Metadata Export
Generation writes `data/processed/labels/augmented_custom_squat_videos_labels.csv` with source/output paths, preserved labels, transformation parameters, video properties, safety status, and review notes.

## 8. Visual QA
Sample source and augmented frames side-by-side before marking outputs usable. Save an optional contact sheet to `reports/figures/augmentation_qa_samples.png`. Confirm the full body remains visible and joint semantics remain interpretable.

In [7]:
# Example QA workflow: load matching frames with cv2, plot with matplotlib, then:
# qa_path = REPO_ROOT / 'reports/figures/augmentation_qa_samples.png'
# qa_path.parent.mkdir(parents=True, exist_ok=True)
# plt.savefig(qa_path, dpi=160, bbox_inches='tight')

## 9. Post-Augmentation Validation
Run:
```powershell
python scripts/prepare_augmented_squat_videos.py
python scripts/extract_landmarks_from_videos.py --metadata data/processed/labels/augmented_custom_squat_videos_labels.csv --input-dir data/augmented/custom_videos --output data/processed/pose_landmarks/augmented_custom_squat_landmarks.csv --failed-output data/processed/pose_landmarks/augmented_failed_videos.csv
python scripts/create_angle_features.py --input data/processed/pose_landmarks/augmented_custom_squat_landmarks.csv --output data/processed/angle_features/augmented_custom_squat_angle_features.csv
```

## 10. Limitations
Augmented videos are correlated variants, not new participants or independent evidence. They may improve robustness testing but cannot solve clinical validation, class imbalance, camera bias, or label uncertainty. More real consented videos and licensed physiotherapist review remain necessary.